In [1]:
import os
from pathlib import Path
from sklearn.model_selection import train_test_split

In [2]:
DATA_DIR = Path("/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC")

files = [str(DATA_DIR.joinpath(f)) for f in os.listdir(DATA_DIR) if f.endswith('.npz')]

# ------------ OUTPUT REPORT ----------
print("Number of files: ", len(files))
print("Files: ", files)

Number of files:  153
Files:  ['/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC/SC4071E0.npz', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC/SC4822G0.npz', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC/SC4352F0.npz', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC/SC4472F0.npz', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC/SC4212E0.npz', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC/SC4011E0.npz', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC/SC4632E0.npz', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC/SC4152E0.npz', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC/SC4171E0.npz', '/home/kasra/courses/2026-edge-computing/project/dataset/sleepEDF/processed/SC/SC4711E0.npz', '/home/kasra/courses/2026-edg

Split dataset

In [3]:
# Get all unique subject IDs from your filenames
subjects = list(map(lambda x: x.split("/")[-1][:5], files))

# Split subjects: 80% train+val, 20% test
train_val_subs, test_subs = train_test_split(subjects, test_size=0.2, random_state=42)

# Split train+val into 70% train, 10% val
train_subs, val_subs = train_test_split(train_val_subs, test_size=0.125, random_state=42)

print(f"Training Subjects: {len(train_subs)} | Val: {len(val_subs)} | Test: {len(test_subs)}")

Training Subjects: 106 | Val: 16 | Test: 31


In [4]:
def subject_to_path(subject_lists, files):
    list_path = []
    for s in subject_lists:
        for f in files:
            if s in f:
                list_path.append(f)
    return list_path
        

In [5]:
train_paths = subject_to_path(train_subs, files)
val_paths = subject_to_path(val_subs, files)
test_paths = subject_to_path(test_subs, files)

In [55]:
import torch
from torch.utils.data import Dataset
import numpy as np

class SleepDataset(Dataset):
    def __init__(self, file_paths: list, window_size=1, trimmed: bool=True):
        self.file_paths = file_paths
        self.window_size = window_size
        self.trimmed = trimmed
        self.data_indeces = self._build_trimmed_index() if trimmed else self._build_index()
        
    def _build_index(self):
        # Maps a global index to (file_path, local_epoch_index)
        index = []
        for path in self.file_paths:
            with np.load(path) as data:
                num_epochs = len(data['y'])
                for i in range(num_epochs - self.window_size + 1):
                    index.append((path, i))
        return index
    
    def _build_trimmed_index(self):
        index = []
        for path in self.file_paths:
            with np.load(path, mmap_mode='r') as data:
                labels = data['y']
                
                # Find the true start (first sleep) and true end (last sleep)
                sleep_mask = np.where(labels != 0)[0]
                if len(sleep_mask) == 0: continue
                
                first_sleep = sleep_mask[0]
                last_sleep = sleep_mask[-1]
                
                # Apply the 30-minute buffer (60 epochs)
                start_search = max(0, first_sleep - 60)
                end_search = min(len(labels), last_sleep + 60)
                
                # Add ALL indices in this range, including the 0s in the middle
                for i in range(start_search, end_search - self.window_size + 1):
                    index.append((path, i))
        return index
    
    def __len__(self):
        return len(self.data_indeces)
    
    def __getitem__(self, idx):
        file_path, start_pos = self.data_indeces[idx]
        with np.load(file_path) as data:
            x = data['x'][start_pos : start_pos + self.window_size]
            mid_idx = start_pos + (self.window_size // 2)
            y = data['y'][mid_idx]
            
            # Z-score Normalization (per-subject/per-file)
            x = (x - np.mean(x)) / (np.std(x) + 1e-8)
        return torch.from_numpy(x).float(), torch.tensor(y).long()
    

In [50]:
with np.load(test_paths[0]) as test_temp:
    tl = test_temp['y']
    for ind, val in enumerate(tl):
        with open("temp_label.txt", 'a') as of:
            of.write(f"ind: {ind} - val: {val}\n")

In [56]:

win_size = 5
train_dataset = SleepDataset(file_paths=train_paths, window_size=win_size, trimmed=True)
test_dataset = SleepDataset(file_paths=test_paths, window_size=win_size, trimmed=True)
val_dataset = SleepDataset(file_paths=val_paths, window_size=win_size, trimmed=True)

In [42]:
x, y = test_dataset[0]
y

tensor(0)

In [ ]:
from torch.utils.data import DataLoader

batch_size = 32
n_workers = 4

train_dl = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=n_workers,
    pin_memory=True
)
val_dl = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=n_workers,
    pin_memory=True
)
test_dl = DataLoader(
    test_dataset, 
    batch_size=batch_size,
    shuffle=False,
    num_workers=n_workers
)

In [ ]:
device = "cude" if torch.cuda.is_available() else "cpu"
